In [1]:
import MyTools as MT
from importlib import reload
import Variable as V
import sqlite3
import pandas as pd
from datetime import datetime, timedelta, date
import Func as F

In [9]:
year_st = 2025
year_en = 2027

current_year = 2999
current_month = 13
current_day = 32
current_hour = 25
columns = ['dt', 'type_rasklada', 'num_palace', 'num_component', 'value']
columns_log = ['dt', 'dt_calculate', 'type_rasklada', 'step', 'txt']
elements = ['戊', '己', '庚', '辛', '壬', '癸', '丁', '丙', '乙']
gate_home = ['杜','景','死','驚','開','休','生','傷']
sky_element = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛', '壬', '癸']
star_home = ['辅', '英', '芮', '柱', '心', '蓬', '任', '冲']
spirit_home_yan = ['天','符','蛇','陰','合','虎','武','地']
spirit_home_yin = ['蛇','符','天','地','武','虎','合','陰']

ERROR! Session/line number was not unique in database. History logging moved to new session 20


In [10]:
def rearrange_sequence(start_num, direction):
    # формируем земную ветвь
    numbers = list(range(1, 10))  # Создаем список от 1 до 9
    
    if direction == 2:
        # Направление прямое Янское
        start_index = numbers.index(start_num)
        return numbers[start_index:] + numbers[:start_index]
    elif direction == 1:
        # Направление обратное Инское
        start_index = numbers.index(start_num)
        return numbers[start_index::-1] + numbers[start_index:][::-1]

In [11]:
def shift_positions(list1, list2):
    start_index = list2.index(next(item for item in list2 if item is not None))  
    position = list2.index(next(item for item in list2 if item is not None)) # Находим индекс заполненной позиции во втором списке
    element = list2[position]
    shift = (start_index - list1.index(list2[position])) % len(list1)  # Вычисляем сдвиг
    #print(f'Сдвиг на {shift} позиций')
    result = [None] * len(list2)
    for i in range(len(list2)):
        if list2[i] == 1:
            result[(i + shift) % len(list2)] = 1
        else:
            result[(i + shift) % len(list2)] = list1[i]

    return result

In [12]:
sql_log = """
CREATE TABLE log (
	dt TEXT,
	dt_calculate TEXT,
	type_rasklada INTEGER,
	step TEXT,
	txt TEXT
);
"""

## Земная тарелка

In [13]:
t_nm_qimen_01 = "t_qimen_01"
sql_t_qimen_01 = """
    CREATE TABLE t_qimen_01 (
id INTEGER PRIMARY KEY AUTOINCREMENT,
	dt TEXT,
    name_hour text,
     god_cha integer,
     mes_cha integer,
    den integer,
    stolp_full text,
    stolp text,
    id_season_start,
    fou_tou text,
    main_gate text,
    main_star text,
	type_rasklada INTEGER,
 earth_1 text, earth_2 text, earth_3 text, earth_4 text, earth_5 text, earth_6 text, earth_7 text, earth_8 text, earth_9 text
, sky_1 TEXT, sky_2 TEXT, sky_3 TEXT, sky_4 TEXT, sky_5 TEXT, sky_6 TEXT, sky_7 TEXT, sky_8 TEXT, sky_9 TEXT
, gate_1 TEXT, gate_2 TEXT, gate_3 TEXT, gate_4 TEXT, gate_5 TEXT, gate_6 TEXT, gate_7 TEXT, gate_8 TEXT, gate_9 TEXT
, star_1 TEXT, star_2 TEXT, star_3 TEXT, star_4 TEXT, star_5 TEXT, star_6 TEXT, star_7 TEXT, star_8 TEXT, star_9 TEXT
, spirit_1 TEXT, spirit_2 TEXT, spirit_3 TEXT, spirit_4 TEXT, spirit_5 TEXT, spirit_6 TEXT, spirit_7 TEXT, spirit_8 TEXT, spirit_9 TEXT
);
"""

In [28]:
# --- Загрузка t_bazci (ваш код без изменений) ---
sql = """
select distinct
  substring(dt,1,10) as dt 
  , name_hour
 , god
 , mes
 , den
 , god_cha
 , mes_cha
, id_season_start
, stolp_goda
, stolp_mes
, stolp_den
, stolp_chas
, god_num_rasklad
, mes_num_rasklad
, den_yin_yan
, den_num_rasklad
, chas_yin_yan
, chas_num_rasklad
from t_bazci

-- Справочник китайских двухчасовок
    left join t_spr_hour spr_hour
    on SUBSTRING(dt, 12,10) = spr_hour.hour

where 1 =1 
 and god_num_rasklad is not NULL   -- Добавлена проверка из вашего фикса
 and mes_num_rasklad is not NULL   -- Добавлена проверка из вашего фикса
 and den_num_rasklad is not NULL   -- Добавлена проверка из вашего фикса
 and chas_num_rasklad is not NULL 
and dt>= date('now')
"""
query = MT.gp_execute(sql_script = sql, rezim = 'list')
df_bazci = pd.DataFrame(query, columns=['dt', 'name_hour','god', 'mes', 'den','god_cha','mes_cha','id_season_start','stolp_goda', 'stolp_mes', 'stolp_den', 'stolp_chas'
                                  , 'god_num_rasklad', 'mes_num_rasklad' ,'den_yin_yan'
                                 ,'den_num_rasklad','chas_yin_yan','chas_num_rasklad'
                                ]) # преобразовали список в dataframe pandas
df_bazci['dt'] = pd.to_datetime(df_bazci['dt'])

df_bazci['id_season_start'] = df_bazci['id_season_start'].fillna(0)
df_bazci['id_season_start'] = df_bazci['id_season_start'].astype(int)

df_bazci['den_yin_yan'] = df_bazci['den_yin_yan'].fillna(0)
df_bazci['den_yin_yan'] = df_bazci['den_yin_yan'].astype(int)

df_bazci['den_num_rasklad'] = df_bazci['den_num_rasklad'].fillna(0)
df_bazci['den_num_rasklad'] = df_bazci['den_num_rasklad'].astype(int)


# --- НОВОЕ: Загружаем справочник Jiazi ОДИН РАЗ ---
sqlfou_tou = """select 
id
, jiazi
, decade
, fo_tuo
from t_spr_jiazi_for_year"""
query_jiazi = MT.gp_execute(sql_script = sqlfou_tou, rezim = 'list')
df_jiazi = pd.DataFrame(query_jiazi, columns=['id', 'jiazi', 'decade', 'fo_tuo'])

In [30]:
# In[15]:

def calculate_chart_data(row, type_rasklada, col_nm, yin_yan, elements, stolp, df_jiazi):
    """
    Расчет данных расклада.
    Не пишет в БД, возвращает словарь с данными и список логов.
    """
    # Список для сбора логов ЭТОГО расчета
    logs = []
    
    # Стартовые приготовления
    col = ['dt', 'name_hour','god_cha', 'mes_cha', 'den', 'stolp_full','stolp','id_season_start' ,'type_rasklada']
    val = [row['dt'].strftime('%Y-%m-%d')
           , row['name_hour']
           , row['god_cha']
           , row['mes_cha']
           , row['dt'].day
           , row['stolp_goda']  + row['stolp_mes']  +  row['stolp_den']  + row['stolp_chas']  
           , stolp
           , row['id_season_start']
           , type_rasklada
          ]

    lo_shu = [4, 9, 2, 7, 6, 1, 8, 3]
    # ****************** Формируем и записываем земную ветвь      
    earth_list = rearrange_sequence(row[col_nm], yin_yan) # годовые расклады всегда инские
    el = [list(tup) for tup in zip(earth_list, elements)]
    for _ in el:
        col.append(f'earth_{_[0]}')
        val.append(_[1])

    # *************** Формируем небесную ветвь
    # --- УДАЛЕНО: SQL-запрос справочника ---
    
    leading_element = stolp[0]
    
    dt_calculate_str = row['dt'].strftime('%Y-%m-%d %H:%M:%S')
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'Стартовый leading_element = {leading_element}'])

    sky_branch = [None, None, None, None, None, None, None, None]
    temp_earth_branch = [None, None, None, None, None, None, None, None]
    gate_branch = [None, None, None, None, None, None, None, None]
    star_branch = [None, None, None, None, None, None, None, None]
    spirit_branch = [None, None, None, None, None, None, None, None]

    sorted_pairs = sorted(zip(earth_list, elements)) 
    earth_branch = [pair[1] for pair in sorted_pairs] 
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f"Получившаяся земная ветвь отсортированная по порядку дворцов earth_branch = [{', '.join(earth_branch)}]"])

    # --- ИЗМЕНЕНО: Берем Fo Tou из DataFrame, а не из БД ---
    fo_tuo = df_jiazi[df_jiazi.jiazi==stolp]['fo_tuo'].iloc[0]
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'Стартовый fo_tuo = {fo_tuo}'])

    if leading_element == '甲':
        leading_element = fo_tuo
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'leading_element 甲 и заменили его на Fou tou, т.е. на заменили на {leading_element}'])
    col.append('fou_tou')
    val.append(fo_tuo)

    for nom, i in enumerate(lo_shu):
        temp_earth_branch[nom] = earth_branch[i-1]

    logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f"""Привели земную ветвь к квадрату Ло Шу temp_earth_branch = [{', '.join(temp_earth_branch)}]"""])

    if leading_element == earth_branch[4]:
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'Лидирующий элемент равен центральному leading_element == earth_branch[4] ==> {leading_element} = {earth_branch[4]}'])
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'Заменили лидирующий элемент на элемент земной ветви 2го дворца {leading_element} --> {earth_branch[1]}'])
        leading_element = earth_branch[1]

    if fo_tuo == earth_branch[4]:
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'fo_tuo равен центральному элементу {fo_tuo} = {earth_branch[4]}. Поэтому перемещать будем Ствол из 2го Дома {temp_earth_branch[2]}'])
        delta = temp_earth_branch.index(leading_element) - temp_earth_branch.index(temp_earth_branch[2])
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'delta  = {leading_element} - {temp_earth_branch[2]} =  {temp_earth_branch.index(leading_element)} - {temp_earth_branch.index(temp_earth_branch[2])} = {delta}'])
    else:
        delta = temp_earth_branch.index(leading_element) - temp_earth_branch.index(fo_tuo) 
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'delta  = {leading_element} - {fo_tuo} =  {temp_earth_branch.index(fo_tuo)} - {temp_earth_branch.index(leading_element)} = {delta}'])
    
    if delta < 0:
            logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f'Delta отрицательная, т.е. дворец лидирующего элемента раньше дворца fo tou. Прибавляем 8 и delta = {delta + 8}'])
            delta = delta + 8

    for i in range(8):
        sky_branch[(i+delta)%8] = temp_earth_branch[i]
    sky_branch = sky_branch + [earth_branch[4]]

    logs.append([now_str, dt_calculate_str, type_rasklada, 'Небесная тарелка', f"""Небесное кольцо = [{', '.join(sky_branch)}]"""])
    el = [list(tup) for tup in zip(lo_shu+[5], sky_branch)]  

    for _ in el:
        col.append(f'sky_{_[0]}')
        val.append(_[1])

    # ********************* Строим кольцо врат ****************************
    if fo_tuo == earth_branch[4]:
        fou_tou_index = 2 
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'Fou tou = {fo_tuo} нет на земном кольце поэтому берем 2й Дом и соответственно 3ю позицию в последовательности'])
    else:
        fou_tou_index = temp_earth_branch.index(fo_tuo)
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f"""Fou tou = {fo_tuo} расположен на земном кольце [{', '.join(temp_earth_branch)}] на {fou_tou_index+1} позиции и соответствует {lo_shu[fou_tou_index]} дворцу"""])

    gate_start = gate_home[fou_tou_index]
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f"""Домашнее расположение ворот [{', '.join(gate_home)}]. На той же позиции {fou_tou_index+1} на домашнем расположении ворот располагаются ворта {gate_start}"""])

    col.append('main_gate')
    val.append(gate_start)    

    poz_leading_element = sky_element.index(stolp[0])+1
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f"""Домашняя последовательность небесных стволов [{', '.join(sky_element)}]. И ведущий элемент {stolp[0]} располагается на {poz_leading_element} позиции"""])

    if type_rasklada in (1, 2) or (type_rasklada == 3 and row['den_yin_yan']==1) or (type_rasklada == 4 and row['chas_yin_yan']==1):
        if fo_tuo == earth_branch[4]:
            num = 5 - poz_leading_element + 1
        else:
            num = lo_shu[fou_tou_index] - poz_leading_element + 1
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'Расклад Инский и последовательность обратная. Новые ворота получились lo_shu[fou_tou_index] - poz_leading_element + 1 = {lo_shu[fou_tou_index]} - {poz_leading_element} + 1= {num}'])
        if num <=0:
            logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'Получилось отрицательное число {num}'])
            num = num + 9
            logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'Новые ворота получились lo_shu[num] = lo_shu[{num}] = {num}'])
    elif (type_rasklada == 3 and row['den_yin_yan'] == 2) or (type_rasklada == 4 and row['chas_yin_yan']==2):
        if fo_tuo == earth_branch[4]:
            num = 5 + poz_leading_element - 1
        else:
            num = lo_shu[fou_tou_index] + poz_leading_element - 1
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'Расклад Янский и последовательность прямая. Новые ворота получились lo_shu[fou_tou_index] + poz_leading_element = {lo_shu[fou_tou_index]} + {poz_leading_element}  = {num}'])
    
    step = None
    if num > 9:
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'т.к. num > 9, то num = num % 9 = {num} % 9 = {num % 9}'])
        num = num % 9
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'num = {num}'])
    if num == 5:
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'Т.к. получился 5й дворец - берем 2й дворец'])
        num = 2
    if num == 0:
         step = 0
         logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'Получился тот же самый дворец, поэтому кольцо ворот не движется от домашнего'])    
    if step == None:
        step = lo_shu.index(num) - fou_tou_index
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f'Итого кольцо ворот движется на lo_shu.index(num) - fou_tou_index = {lo_shu.index(num)} - {fou_tou_index} = {step}'])

    for i in range(8):
        gate_branch[(i+step)%8] = gate_home[i]
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо врат', f"""Итого кольцо ворот получилось: [{', '.join(gate_branch)}]"""])

    el = [list(tup) for tup in zip(lo_shu, gate_branch)]
    for _ in el:
        col.append(f'gate_{_[0]}')
        val.append(_[1])

    # ********************* Строим кольцо Звезд ****************************
    star_poz_st = fou_tou_index
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо звезд', f"""Fou tou = {fo_tuo} расположен на земном кольце [{', '.join(temp_earth_branch)}] на {fou_tou_index+1} позиции и соответствует {lo_shu[fou_tou_index]} дворцу"""])
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо звезд', f"""На домашнем распооложении звезд [{', '.join(star_home)}] в {lo_shu[fou_tou_index]} дворце расположена звезда {star_home[fou_tou_index]}"""])

    col.append('main_star')
    val.append(star_home[fou_tou_index])

    star_poz_en = temp_earth_branch.index(leading_element) 
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо звезд', f"""Лидирующий элемент {leading_element} на земном кольце [{', '.join(temp_earth_branch)}] на {star_poz_en+1} позиции и соответствует {lo_shu[star_poz_en]} дворцу"""])

    step = star_poz_en - star_poz_st
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо звезд', f'Поворот кольца звезд составил star_poz_en - star_poz_st = {star_poz_en} - star_poz_st = {step} шагов'])

    for i in range(8):
        star_branch[(i+step)%8] = star_home[i]
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо звезд', f"""Итого кольцо звезд получилось: [{', '.join(star_branch)}]"""])

    el = [list(tup) for tup in zip(lo_shu, star_branch)]
    for _ in el:
        col.append(f'star_{_[0]}')
        val.append(_[1])

    # ********************* Строим кольцо духов ****************************
    spirit_st = 1 
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо духов', f'Домашня позиция Джи Фу всегда 9й дворец (или 2я позиция по квадрату Ло шу) spirit_st = 2 '])
    spirit_en = temp_earth_branch.index(leading_element)
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо духов', f"""Джи Фу помещается в дворец {lo_shu[temp_earth_branch.index(leading_element)]} где располагается ведущего элемента {leading_element} на земном кольце [{', '.join(temp_earth_branch)}] """])

    step = spirit_en - spirit_st
    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо духов', f'Шаг поворота кольца духов = spirit_en - spirit_st= {spirit_en+1} - {spirit_st+1} = {step} '])

    if type_rasklada in (1, 2) or (type_rasklada == 3 and row['den_yin_yan']==1) or (type_rasklada == 4 and row['chas_yin_yan']==1): #Инская последовательность
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо духов', f"""Т.к. расклад Инский, то используется Инская домашняя последовательность духов: [{', '.join(spirit_home_yin)}]"""])
        for i in range(8):
            spirit_branch[(i+step)%8] = spirit_home_yin[i]
    elif (type_rasklada == 3 and row['den_yin_yan']==2) or (type_rasklada == 4 and row['chas_yin_yan']==2): # Янская последовательность
        logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо духов', f"""Т.к. расклад Янский, то используется Янская домашняя последовательность духов: [{', '.join(spirit_home_yin)}]"""])
        for i in range(8):
            spirit_branch[(i+step)%8] = spirit_home_yan[i]

    logs.append([now_str, dt_calculate_str, type_rasklada, 'Кольцо духов', f"""Итого кольцо духов получилось: [{', '.join(star_branch)}]"""])

    el = [list(tup) for tup in zip(lo_shu, spirit_branch)]
    for _ in el:
        col.append(f'spirit_{_[0]}')
        val.append(_[1])
    
    # --- УДАЛЕНО: MT.insert_data(tb = tb_nm, column = col, value = val) ---
    
    # --- НОВОЕ: Возвращаем словарь с данными и список логов ---
    return dict(zip(col, val)), logs

In [31]:
# In[16]:

print(f"Загружено {len(df_bazci)} строк для расчета.")

# Списки для СБОРА всех результатов и логов В ПАМЯТИ
all_results_wide = []
all_logs = []

current_year = 2999
current_month = 13
current_day = 32

for index, row in df_bazci.iterrows():
    if row['god_cha'] !=  current_year:
        if row['god_num_rasklad'] != 0: # Проверка из фикса
            # годовой расклад
            res_dict, logs_list = calculate_chart_data(row=row, type_rasklada=1, col_nm='god_num_rasklad', yin_yan=1, elements=elements, stolp=row['stolp_goda'], df_jiazi=df_jiazi)
            all_results_wide.append(res_dict)
            all_logs.extend(logs_list)
        current_year = row['god_cha']

    if row['mes_cha'] !=  current_month:
        if row['mes_num_rasklad'] != 0: # Проверка из фикса
            #месячный расклад
            res_dict, logs_list = calculate_chart_data(row=row, type_rasklada=2, col_nm='mes_num_rasklad', yin_yan=1, elements=elements, stolp=row['stolp_mes'], df_jiazi=df_jiazi)
            all_results_wide.append(res_dict)
            all_logs.extend(logs_list)
        current_month = row['mes_cha']

    if row['dt'].day !=  current_day:
        if row['den_num_rasklad'] != 0: # Проверка из фикса
            #дневной расклад
            res_dict, logs_list = calculate_chart_data(row=row, type_rasklada=3, col_nm='den_num_rasklad', yin_yan=row['den_yin_yan'], elements=elements, stolp=row['stolp_den'], df_jiazi=df_jiazi)
            all_results_wide.append(res_dict)
            all_logs.extend(logs_list)
        current_day = row['dt'].day

    #Часовой расклад
    if row['chas_num_rasklad'] != 0: # Проверка из фикса
        res_dict, logs_list = calculate_chart_data(row=row, type_rasklada=4, col_nm='chas_num_rasklad', yin_yan=row['chas_yin_yan'], elements=elements, stolp=row['stolp_chas'], df_jiazi=df_jiazi)
        all_results_wide.append(res_dict)
        all_logs.extend(logs_list)

print(f"Расчеты в памяти завершены. Собрано {len(all_results_wide)} раскладов и {len(all_logs)} записей логов.")

Загружено 19687 строк для расчета.
Расчеты в памяти завершены. Собрано 21255 раскладов и 508542 записей логов.


In [ ]:
rasklad = [1,2,3,4]
dt = ['2024-07-06']
chas = ['Ранняя крыса', 'Бык', 'Тигр', 'Кролик', 'Дракон', 'Змея', 'Лошадь', 'Коза', 'Обезьяна', 'Петух', 'Собака', 'Свинья', 'Поздняя крыса']

for r in rasklad:
    for d in dt:
        for c in chas:
            try:
                sql_poisk = f"""
                 select 
                id
                , vh.dt
                , vh.name_hour
                , spr_hour.moscow_hour_recomend
                
                , vh.god_cha
                , mes_cha
                , den
                , stolp
       
                , fou_tou
                , type_rasklada
                , earth_1 , earth_2 , earth_3 , earth_4 , earth_5 , earth_6 , earth_7 , earth_8, earth_9 
                , sky_1, sky_2, sky_3, sky_4, sky_5, sky_6, sky_7, sky_8, sky_9
                , gate_1, gate_2, gate_3, gate_4, gate_5, gate_6, gate_7, gate_8, gate_9
                , star_1, star_2, star_3, star_4, star_5, star_6, star_7, star_8, star_9
                , spirit_1, spirit_2, spirit_3, spirit_4, spirit_5, spirit_6, spirit_7, spirit_8, spirit_9
                from t_qimen_01 vh

                -- Справочник китайских двухчасовок
                    left join t_spr_hour spr_hour
                    on vh.name_hour = spr_hour.name_hour


                where 1 = 1 
                    and dt = '{d}'
                    and vh.name_hour = '{c}'
                """
                query = MT.gp_execute(sql_script = sql_poisk, rezim = 'list')
                df = pd.DataFrame(query, columns=[
                    'id'
                , 'dt'
                , 'name_hour'
                , 'moscow_hour_recomend'
                , 'god_cha'
                , 'mes_cha'
                , 'den'
                , 'stolp'
                , 'fou_tou'
                , 'type_rasklada'
                , 'earth_1', 'earth_2', 'earth_3', 'earth_4', 'earth_5', 'earth_6', 'earth_7', 'earth_8', 'earth_9'
                , 'sky_1', 'sky_2', 'sky_3', 'sky_4', 'sky_5', 'sky_6', 'sky_7', 'sky_8', 'sky_9'
                , 'gate_1', 'gate_2', 'gate_3', 'gate_4', 'gate_5', 'gate_6', 'gate_7', 'gate_8', 'gate_9'
                , 'star_1', 'star_2', 'star_3', 'star_4', 'star_5', 'star_6', 'star_7', 'star_8', 'star_9'
                , 'spirit_1', 'spirit_2', 'spirit_3', 'spirit_4', 'spirit_5', 'spirit_6', 'spirit_7', 'spirit_8', 'spirit_9'
                                                 ]) # преобразовали список в dataframe pandas
                
                if df.type_rasklada.iloc[0] == 1:
                    type_rasklada = "Годовой расклад"
                elif df.type_rasklada.iloc[0] == 2:
                    type_rasklada = "Месячный расклад"
                elif df.type_rasklada.iloc[0] == 3:
                    type_rasklada = "Дневной расклад"
                elif df.type_rasklada.iloc[0] == 4:
                    type_rasklada = "Часовой расклад"
                
                shablon = f"""
                Дата: {df.dt.iloc[0]}
                Время: {df.name_hour.iloc[0]} ({df.moscow_hour_recomend.iloc[0]})
                Расклад: {type_rasklada}
                Столп:{df.stolp.iloc[0]} 
                ----------------------------------
                ||{df.spirit_4.iloc[0]}  {df.gate_4.iloc[0]}  {df.sky_4.iloc[0]}||{df.spirit_9.iloc[0]}  {df.gate_9.iloc[0]}  {df.sky_9.iloc[0]}||{df.spirit_2.iloc[0]}  {df.gate_2.iloc[0]}  {df.sky_2.iloc[0]}||
                ||{df.star_4.iloc[0]} -4-  {df.earth_4.iloc[0]}||{df.star_9.iloc[0]} -9-  {df.earth_9.iloc[0]}||{df.star_2.iloc[0]} -2- {df.earth_2.iloc[0]}||
                ----------------------------------
                ||{df.spirit_3.iloc[0]}  {df.gate_3.iloc[0]}  {df.sky_3.iloc[0]}||{df.spirit_3.iloc[0]} {df.gate_5.iloc[0]} {df.sky_5.iloc[0]}||{df.spirit_7.iloc[0]}  {df.gate_7.iloc[0]}  {df.sky_7.iloc[0]}||
                ||{df.star_3.iloc[0]} -3-  {df.earth_3.iloc[0]}||{df.star_5.iloc[0]} -- {df.earth_5.iloc[0]}||{df.star_7.iloc[0]} -7- {df.earth_7.iloc[0]}||
                ----------------------------------
                ||{df.spirit_8.iloc[0]}  {df.gate_8.iloc[0]}  {df.sky_8.iloc[0]}||{df.spirit_1.iloc[0]}  {df.gate_1.iloc[0]}  {df.sky_1.iloc[0]}||{df.spirit_6.iloc[0]}  {df.gate_6.iloc[0]}  {df.sky_6.iloc[0]}||
                ||{df.star_8.iloc[0]} -8-  {df.earth_8.iloc[0]}||{df.star_1.iloc[0]} -1-  {df.earth_1.iloc[0]}||{df.star_6.iloc[0]} -6- {df.earth_6.iloc[0]}||
                """
                print(shablon)
            except Exception:
                print(sql_poisk)
                pass

In [18]:
p = [1, 2,3,4,5,6,7,8,9]
tb_nm = 't_qimen_02'
for num in p:
    sql = f"""
    select distinct
    SUBSTRING(vh.dt, 1,10) ||  spr_hour.name_hour || vh.type_rasklada || vh.fou_tou || vh.main_gate || vh.main_star as id_rasklada
    , SUBSTRING(vh.dt, 1,10) as dt
    , spr_hour.name_hour
    , spr_hour.moscow_hour_st
    , spr_hour.moscow_hour_en
    , spr_hour.moscow_hour_recomend
    , vh.god_cha
    , vh.mes_cha
    , vh.den

    , substring(vh.stolp_full, 1,2) as stolp_goda
    , substring(vh.stolp_full, 3,2) as stolp_mes
    , substring(vh.stolp_full, 5,2) as stolp_den
    , substring(vh.stolp_full, 7,2) as stolp_chas

    , substring(vh.stolp_full, 1,1) as god_stolp_sky
    , substring(vh.stolp_full, 2,1) as god_stolp_earth
    , substring(vh.stolp_full, 3,1) as mes_stolp_sky
    , substring(vh.stolp_full, 4,1) as mes_stolp_earth
    , substring(vh.stolp_full, 5,1) as den_stolp_sky
    , substring(vh.stolp_full, 6,1) as den_stolp_earth
    , substring(vh.stolp_full, 7,1) as chas_stolp_sky
    , substring(vh.stolp_full, 8,1) as chas_stolp_earth
    
    , vh.fou_tou
    , vh.main_gate
    , vh.main_star
    
    -- столп и фаза ци
    , vh.stolp_full
    , vh.stolp
    
    -- сезон
    , vh.id_season_start

    -- Расклад
    , vh.type_rasklada
    , spr_rasklada.nm_rasklada

    -- Дворец 
    , {num} as palace 
    , spr_palace.direction
        
    -- ворота
    , vh.gate_{num} as gate
    , case when vh.gate_{num} = vh.main_gate then 1 end as flag_main_gate
    , spr_gate.gate_rus_2 as gate_nm
   
    --структура
    , vh.sky_{num} as sky
    , case when vh.sky_{num} = vh.fou_tou then 1 end as flag_sky_fou_tou
    , case when vh.earth_{num} = vh.fou_tou then 1 end as flag_earth_fou_tou
    , vh.earth_{num} as earth
    , vh.sky_{num} || vh.earth_{num} as structure
    
     -- звезда
    , vh.star_{num} as star
    , case when vh.star_{num} = vh.main_star then 1 end as flag_main_star
    
    -- дух
    , vh.spirit_{num} as spirit
   
    from t_qimen_01 vh

    -- Справочник раскладов
    left join t_spr_rasklada spr_rasklada
    on vh.type_rasklada = spr_rasklada.id

    
    -- Справочник китайских двухчасовок
    left join t_spr_hour spr_hour
    on vh.name_hour = spr_hour.name_hour

     -- Справочник дворца
    left join t_spr_palace spr_palace
    on {num} = spr_palace.id

    -- справочник ворот
    left join t_spr_gate spr_gate
    on  vh.gate_{num} = spr_gate.gate_cha 

    WHERE 1 = 1 
    	and vh.dt>= date('now')
    """
    if num == 1:
        MT.drop_table(tb_nm)
        MT.create_table(tb_target = tb_nm, sql_script = sql)
        MT.gp_execute(sql_script = sql)
    else:
        sql2 = f"""insert into {tb_nm} 
        {sql}"""
        MT.gp_execute(sql_script = sql2)

Таблица t_qimen_02 успешно удалена
Таблица t_qimen_02 Успешно создана
Скрипт create_table выполнился за 0 дней 0 часов 0 минут 0 секунд (с 2025-11-01 18:34:10 по 2025-11-01 18:34:10)


In [42]:
# In[17]:

import sqlite3
# import Variable as V # У вас он уже импортирован
# import MyTools as MT # У вас он уже импортирован

# --- 1. Создаем DataFrame'ы (этот код у вас уже есть) ---
print("Создание DataFrame'ов...")
df_qimen_01 = pd.DataFrame(all_results_wide)
df_logs = pd.DataFrame(all_logs, columns=columns_log)

# --- 2. Пакетная запись в БД с учетом CHUNKSIZE ---
print("Начало пакетной записи в БД...")

# Лимит переменных в SQLite
SQLITE_MAX_VARS = 999

# Рассчитываем безопасный chunksize для каждого DataFrame
# (Лимит / кол-во столбцов)
chunksize_logs = 0
if len(df_logs.columns) > 0:
    chunksize_logs = SQLITE_MAX_VARS // len(df_logs.columns)
    print(f"Рассчитан chunksize для 'log' ({len(df_logs.columns)} столбцов): {chunksize_logs}")

chunksize_qimen = 0
if len(df_qimen_01.columns) > 0:
    chunksize_qimen = SQLITE_MAX_VARS // len(df_qimen_01.columns)
    print(f"Рассчитан chunksize для 't_qimen_01' ({len(df_qimen_01.columns)} столбцов): {chunksize_qimen}")

try:
    # !!! УКАЖИТЕ ПУТЬ К ВАШЕЙ БД !!!
    conn = sqlite3.connect(V.bd_name) # <-- Убедитесь, что V.bd_name верное
    
    # --- Запись логов (с chunksize) ---
    MT.drop_table('log')
    MT.gp_execute(sql_log) # sql_log из Ячейки [12]
    
    # Запускаем to_sql только если DataFrame не пустой
    if not df_logs.empty and chunksize_logs > 0:
        df_logs.to_sql(
            'log', 
            conn, 
            if_exists='append', 
            index=False, 
            method='multi',
            chunksize=chunksize_logs  # <-- РЕШЕНИЕ
        )
    print(f"Запись {len(df_logs)} логов завершена.")

    # --- Запись t_qimen_01 (с chunksize) ---
    MT.drop_table(t_nm_qimen_01) # t_nm_qimen_01 из Ячейки [13]
    MT.gp_execute(sql_t_qimen_01) # sql_t_qimen_01 из Ячейки [13]
    
    # Запускаем to_sql только если DataFrame не пустой
    if not df_qimen_01.empty and chunksize_qimen > 0:
        df_qimen_01.to_sql(
            t_nm_qimen_01, 
            conn, 
            if_exists='append', 
            index=False, 
            method='multi',
            chunksize=chunksize_qimen # <-- РЕШЕНИЕ
        )
    print(f"Запись {len(df_qimen_01)} строк в t_qimen_01 завершена.")

except Exception as e:
    print(f"Ошибка при пакетной записи: {e}")
finally:
    if 'conn' in locals() and conn:
        conn.close() # Важно закрыть подключение

print("Пакетная запись в t_qimen_01 и log завершена.")

Создание DataFrame'ов...
Начало пакетной записи в БД...
Рассчитан chunksize для 'log' (5 столбцов): 199
Рассчитан chunksize для 't_qimen_01' (54 столбцов): 18
Таблица log успешно удалена
Запись 508542 логов завершена.
Таблица t_qimen_01 успешно удалена
Запись 21255 строк в t_qimen_01 завершена.
Пакетная запись в t_qimen_01 и log завершена.


In [43]:
# In[18]:

print("Начало трансформации (unpivot) данных в Pandas...")

# --- 1. Загружаем все справочники в Pandas ---
#    Нам нужно снова подключиться к БД для чтения
try:
    # !!! УКАЖИТЕ ПУТЬ К ВАШЕЙ БД !!!
    # conn = sqlite3.connect(V.DB_NAME) 
    conn = sqlite3.connect('ваша_база_данных.db') # <-- ЗАМЕНИТЕ ЭТО

    df_spr_hour = pd.read_sql_query("SELECT * FROM t_spr_hour", conn)
    df_spr_rasklada = pd.read_sql_query("SELECT * FROM t_spr_rasklada", conn)
    df_spr_palace = pd.read_sql_query("SELECT * FROM t_spr_palace", conn)
    df_spr_gate = pd.read_sql_query("SELECT * FROM t_spr_gate", conn)
    
except Exception as e:
    print(f"Ошибка при чтении справочников: {e}")
finally:
    if 'conn' in locals() and conn:
        conn.close()

print("Справочники загружены в Pandas.")

# --- 2. Определяем "ключевые" колонки (которые не будут разворачиваться) ---
id_vars = ['dt', 'name_hour', 'god_cha', 'mes_cha', 'den', 'stolp_full', 'stolp',
           'id_season_start', 'type_rasklada', 'fou_tou', 'main_gate', 'main_star']

# --- 3. "Разворачиваем" (unpivot) df_qimen_01 из широкого в длинный ---
#    Мы используем pd.melt для каждого типа компонента (earth, sky, ...)
#    а затем соединяем их вместе.

dfs_to_merge = []
for component in ['earth', 'sky', 'gate', 'star', 'spirit']:
    # Выбираем колонки для melt: id_vars + колонки этого компонента
    cols_to_melt = [f'{component}_{i}' for i in range(1, 10)]
    
    # Выполняем melt
    df_melted = df_qimen_01[id_vars + cols_to_melt].melt(
        id_vars=id_vars,
        value_vars=cols_to_melt,
        var_name='component_palace',
        value_name=component # Имя колонки = имя компонента
    )
    
    # Из 'earth_1' получаем '1' (дворец)
    df_melted['palace'] = df_melted['component_palace'].str.split('_').str[1].astype(int)
    df_melted = df_melted.drop(columns=['component_palace'])
    
    # Устанавливаем ключи для будущего соединения
    df_melted = df_melted.set_index(id_vars + ['palace'])
    dfs_to_merge.append(df_melted)

# Соединяем все (earth, sky, gate...) по ключам (id_vars + palace)
print("Соединение развернутых компонентов...")
df_long = pd.concat(dfs_to_merge, axis=1).reset_index()

print(f"Создана 'длинная' таблица из {len(df_long)} строк.")

# --- 4. Обогащаем данными из справочников (JOINs) ---
print("Обогащение данными из справочников (JOINs)...")

# Добавляем id_rasklada
df_long['id_rasklada'] = df_long['dt'].astype(str) + df_long['name_hour'] + \
                         df_long['type_rasklada'].astype(str) + df_long['fou_tou'] + \
                         df_long['main_gate'] + df_long['main_star']

# Добавляем данные по часам (t_spr_hour)
df_final = pd.merge(df_long, df_spr_hour, on='name_hour', how='left')

# Добавляем данные по раскладам (t_spr_rasklada)
df_final = pd.merge(df_final, df_spr_rasklada, left_on='type_rasklada', right_on='id', how='left')

# Добавляем данные по дворцам (t_spr_palace)
df_final = pd.merge(df_final, df_spr_palace, left_on='palace', right_on='id', how='left')

# Добавляем данные по воротам (t_spr_gate)
df_final = pd.merge(df_final, df_spr_gate[['gate_cha', 'gate_rus_2']], left_on='gate', right_on='gate_cha', how='left')
df_final = df_final.rename(columns={'gate_rus_2': 'gate_nm'})

# --- 5. Добавляем вычисляемые флаги и столбцы ---
print("Добавление вычисляемых флагов...")

# Флаги
df_final['flag_main_gate'] = (df_final['gate'] == df_final['main_gate']).astype(int)
df_final['flag_main_star'] = (df_final['star'] == df_final['main_star']).astype(int)
df_final['flag_sky_fou_tou'] = (df_final['sky'] == df_final['fou_tou']).astype(int)
df_final['flag_earth_fou_tou'] = (df_final['earth'] == df_final['fou_tou']).astype(int)

# Структура
df_final['structure'] = df_final['sky'] + df_final['earth']

# Разделение столпов
df_final['stolp_goda'] = df_final['stolp_full'].str.slice(0, 2)
df_final['stolp_mes'] = df_final['stolp_full'].str.slice(2, 4)
df_final['stolp_den'] = df_final['stolp_full'].str.slice(4, 6)
df_final['stolp_chas'] = df_final['stolp_full'].str.slice(6, 8)

df_final['god_stolp_sky'] = df_final['stolp_full'].str.slice(0, 1)
df_final['god_stolp_earth'] = df_final['stolp_full'].str.slice(1, 2)
df_final['mes_stolp_sky'] = df_final['stolp_full'].str.slice(2, 3)
df_final['mes_stolp_earth'] = df_final['stolp_full'].str.slice(3, 4)
df_final['den_stolp_sky'] = df_final['stolp_full'].str.slice(4, 5)
df_final['den_stolp_earth'] = df_final['stolp_full'].str.slice(5, 6)
df_final['chas_stolp_sky'] = df_final['stolp_full'].str.slice(6, 7)
df_final['chas_ stolp_earth'] = df_final['stolp_full'].str.slice(7, 8) # Опечатка в вашем SQL (chas_), исправлено

# --- 6. Выбираем и упорядочиваем финальные колонки ---
#    (Берем названия из вашего sql_create_table в Ячейке [19])
final_columns = [
    'id_rasklada', 'dt', 'name_hour', 'moscow_hour_st', 'moscow_hour_en', 'moscow_hour_recomend',
    'god_cha', 'mes_cha', 'den', 'stolp_goda', 'stolp_mes', 'stolp_den', 'stolp_chas',
    'god_stolp_sky', 'god_stolp_earth', 'mes_stolp_sky', 'mes_stolp_earth',
    'den_stolp_sky', 'den_stolp_earth', 'chas_stolp_sky', 'chas_ stolp_earth', # и здесь
    'fou_tou', 'main_gate', 'main_star', 'stolp_full', 'stolp', 'id_season_start',
    'type_rasklada', 'nm_rasklada', 'palace', 'direction', 'gate', 'flag_main_gate',
    'gate_nm', 'sky', 'flag_sky_fou_tou', 'flag_earth_fou_tou', 'earth', 'structure',
    'star', 'flag_main_star', 'spirit'
]

# Убираем лишние 'id' от справочников и gate_cha
df_to_write = df_final[final_columns]

print(f"Трансформация завершена. Готово к записи {len(df_to_write)} строк в t_qimen.")

# --- 7. Пакетная запись финальной таблицы t_qimen ---
tb_nm = 't_qimen'
sql_create_table = """
CREATE TABLE t_qimen(
  id_rasklada TEXT, dt TEXT, name_hour TEXT, moscow_hour_st TEXT, moscow_hour_en TEXT,
  moscow_hour_recomend TEXT, god_cha INT, mes_cha INT, den INT, stolp_goda TEXT,
  stolp_mes TEXT, stolp_den TEXT, stolp_chas TEXT, god_stolp_sky TEXT,
  god_stolp_earth TEXT, mes_stolp_sky TEXT, mes_stolp_earth TEXT, den_stolp_sky TEXT,
  den_stolp_earth TEXT, chas_stolp_sky TEXT, "chas_ stolp_earth" TEXT, fou_tou TEXT, -- Обратите внимание на кавычки
  main_gate TEXT, main_star TEXT, stolp_full TEXT, stolp TEXT, id_season_start int,
  type_rasklada INT, nm_rasklada TEXT, palace int, direction TEXT, gate TEXT,
  flag_main_gate int, gate_nm TEXT, sky TEXT, flag_sky_fou_tou int,
  flag_earth_fou_tou int, earth TEXT, structure text, star TEXT, flag_main_star int,
  spirit TEXT
);"""

try:
    # !!! УКАЖИТЕ ПУТЬ К ВАШЕЙ БД !!!
    # conn = sqlite3.connect(V.DB_NAME) 
    conn = sqlite3.connect(V.bd_name) # <-- ЗАМЕНИТЕ ЭТО
    
    MT.drop_table(tb_nm)
    # Используем MT.gp_execute, так как он у вас уже есть
    conn.execute(sql_create_table) 
    conn.commit()

    # Пакетная запись
    df_to_write.to_sql(tb_nm, conn, if_exists='append', index=False, method='multi')
    
    print("Финальная таблица t_qimen успешно создана и заполнена.")

except Exception as e:
    print(f"Ошибка при записи t_qimen: {e}")
finally:
    if 'conn' in locals() and conn:
        conn.close()

Начало трансформации (unpivot) данных в Pandas...
Ошибка при чтении справочников: Execution failed on sql 'SELECT * FROM t_spr_hour': no such table: t_spr_hour
Справочники загружены в Pandas.


KeyError: "['gate_5'] not in index"

In [24]:
import nbformat
from nbconvert import PythonExporter

# Путь к текущему блокноту
nm = 'Qimen 29'
notebook_path = f'{nm}.ipynb'  # ← заменить на имя твоего файла

# Чтение блокнота
with open(notebook_path, 'r', encoding='utf-8') as f:
    notebook = nbformat.read(f, as_version=4)

# Экспорт в .py
exporter = PythonExporter()
script, _ = exporter.from_notebook_node(notebook)

# Сохранение
output_file = f'{nm}.py'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(script)

print(f"Файл успешно сохранён как {output_file}")

Файл успешно сохранён как Qimen 29.py
